In [0]:
import os
from datetime import datetime

# Local ephemeral storage path on the driver node
today_str = datetime.now().strftime('%Y-%m-%d')
base_raw_path = f"/tmp/fhir_raw/{today_str}"

resources = ["Patient", "Encounter", "Observation", "Condition"]

for resource in resources:
    dir_path = os.path.join(base_raw_path, resource)
    os.makedirs(dir_path, exist_ok=True)

print(f"Local raw directories created successfully at: {base_raw_path}/")

✅ Local raw directories created successfully at: /tmp/fhir_raw/2026-07-12/


In [0]:
import requests
import json
import os
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp, lit, from_json, schema_of_json

def fetch_and_ingest_fhir(resource_name, days_back=3):
    start_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    base_url = f"https://hapi.fhir.org/baseR4/{resource_name}"
    url = f"{base_url}?_lastUpdated=ge{start_date}&_count=100"
    
    all_entries = []
    
    print(f"Processing Resource: {resource_name} (Days Back: {days_back})")
    
    # Incremental API Fetch with Pagination
    while url:
        try:
            response = requests.get(url, timeout=30)
            if response.status_code != 200:
                print(f"Error fetching {url}: Status {response.status_code}")
                break
            
            data = response.json()
            if "entry" in data:
                all_entries.extend([item["resource"] for item in data["entry"]])
                
            next_link = next((link["url"] for link in data.get("link", []) if link.get("relation") == "next"), None)
            url = next_link
        except Exception as e:
            print(f"Error during pagination: {e}")
            break

    print(f"Retrieved {len(all_entries)} records for {resource_name}.")
    if not all_entries:
        return

    # Store Raw JSON locally (Metadata Archiving)
    today_str = datetime.now().strftime('%Y-%m-%d')
    local_dir_path = f"/tmp/fhir_raw/{today_str}/{resource_name}"
    os.makedirs(local_dir_path, exist_ok=True)
    
    local_file_path = os.path.join(local_dir_path, f"{resource_name}_raw.json")
    with open(local_file_path, "w") as f:
        json.dump(all_entries, f)

    # Dynamic Schema Inference & Parsing
    json_tuples = [(json.dumps(e),) for e in all_entries]
    df_raw_str = spark.createDataFrame(json_tuples, ["raw_json"])
    sample_json = json_tuples[0][0]
    inferred_schema = schema_of_json(sample_json)
    
    df_parsed = df_raw_str.withColumn("data", from_json("raw_json", inferred_schema)).select("data.*")

    # Save Raw Table
    raw_table_name = f"raw.{resource_name.lower()}"
    df_parsed.write.format("delta").mode("overwrite").saveAsTable(raw_table_name)

    # Save Bronze Table with Ingestion Metadata
    df_bronze = df_parsed.withColumn("extraction_timestamp", current_timestamp()) \
                         .withColumn("api_url_or_params", lit(f"{base_url}?_lastUpdated=ge{start_date}"))

    bronze_table_name = f"bronze.{resource_name.lower()}"
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_table_name)
    print(f"Successfully ingested & logged {resource_name} into {raw_table_name} & {bronze_table_name}")


# DYNAMIC METADATA-DRIVEN EXECUTION LOOP
metadata_df = spark.sql("""
    SELECT resource_name, days_back 
    FROM raw.ingestion_metadata 
    WHERE is_active = true 
    ORDER BY execution_order ASC
""").collect()

for row in metadata_df:
    fetch_and_ingest_fhir(resource_name=row["resource_name"], days_back=row["days_back"])

Fetching Patient starting from 2026-07-09...
Retrieved 653 records for Patient.
Saved local raw JSON file to /tmp/fhir_raw/2026-07-12/Patient/Patient_raw.json
Saved Delta tables: raw.patient & bronze.patient
--------------------------------------------------
Fetching Encounter starting from 2026-07-09...
Retrieved 2696 records for Encounter.
Saved local raw JSON file to /tmp/fhir_raw/2026-07-12/Encounter/Encounter_raw.json
Saved Delta tables: raw.encounter & bronze.encounter
--------------------------------------------------
Fetching Observation starting from 2026-07-09...
Retrieved 10526 records for Observation.
Saved local raw JSON file to /tmp/fhir_raw/2026-07-12/Observation/Observation_raw.json
Saved Delta tables: raw.observation & bronze.observation
--------------------------------------------------
Fetching Condition starting from 2026-07-09...
Retrieved 2124 records for Condition.
Saved local raw JSON file to /tmp/fhir_raw/2026-07-12/Condition/Condition_raw.json
Saved Delta tabl